# Time-series baseline: ARIMA on log returns
Select order using training AIC, compare one-step forecasts against zero returns on a chronological holdout. Returns use consecutive observed records for each equity; intervals may span missing exchange sessions. Models describe observed-event time, not necessarily calendar days. ADF/KPSS results inform differencing; returns are modelled with d=0 as a working baseline.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.stattools import adfuller,kpss
import matplotlib.pyplot as plt
root=Path.cwd() if (Path.cwd()/'data').exists() else Path.cwd().parent
out=root/'outputs/tables'; figs=root/'outputs/figures/arima';figs.mkdir(parents=True,exist_ok=True)
orders=[]; results=[]
for a in ['JKH','HAYL','VONE']:
    f=pd.read_csv(root/f'data/processed/{a}_cleaned.csv',parse_dates=['Date']).sort_values('Date')
    r=np.log(f.Adjusted_Close).diff().dropna().to_numpy()*100
    cut=int(.8*len(r));train=r[:cut];test=r[cut:]; fits=[]
    for p,q in [(0,0),(1,0),(0,1),(1,1),(2,0),(0,2)]:
        fit=ARIMA(train,order=(p,0,q)).fit();ok=bool(fit.mle_retvals.get('converged',False))
        orders.append({'Company':a,'p':p,'d':0,'q':q,'AIC':fit.aic,'BIC':fit.bic,'Converged':ok})
        if ok:fits.append(fit)
    best=min(fits,key=lambda x:x.aic); p,d,q=best.model.order
    extended=best.append(test,refit=False); pred=np.asarray(extended.fittedvalues)[cut:]
    # Each test prediction uses only earlier observations; parameters remain fixed.
    for label,yhat in [('ARIMA',pred),('Zero return',np.zeros_like(test))]:
        results.append({'Company':a,'Method':label,'Order':str((p,d,q)),'Test N':len(test),'MAE':np.mean(abs(test-yhat)),'RMSE':np.sqrt(np.mean((test-yhat)**2)),'Training ADF p':adfuller(train)[1],'Training KPSS p':kpss(train,nlags='auto')[1],'Residual Ljung Box p':acorr_ljungbox(best.resid,lags=[10],model_df=p+q,return_df=True).lb_pvalue.iloc[0],'Residual ARCH LM p':het_arch(best.resid,nlags=10)[1]})
    dates=f.Date.iloc[1+cut:];fig,ax=plt.subplots(figsize=(11,4));ax.plot(dates,test,label='Actual',alpha=.6);ax.plot(dates,pred,label='One-step forecast');ax.set(ylabel='Log return (%)',title=a);ax.legend();fig.tight_layout();fig.savefig(figs/f'{a}.png',dpi=150);plt.close(fig)
for name,rows in [('arima_order_selection',orders),('arima_validation',results)]:
    t=pd.DataFrame(rows);t.to_csv(out/f'{name}.csv',index=False);display(t)


/tmp/ipykernel_835/984586289.py:23: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  results.append({'Company':a,'Method':label,'Order':str((p,d,q)),'Test N':len(test),'MAE':np.mean(abs(test-yhat)),'RMSE':np.sqrt(np.mean((test-yhat)**2)),'Training ADF p':adfuller(train)[1],'Training KPSS p':kpss(train,nlags='auto')[1],'Residual Ljung Box p':acorr_ljungbox(best.resid,lags=[10],model_df=p+q,return_df=True).lb_pvalue.iloc[0],'Residual ARCH LM p':het_arch(best.resid,nlags=10)[1]})
/tmp/ipykernel_835/984586289.py:23: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  results.append({'Company':a,'Method':label,'Order':str((p,d,q)),'Test N':len(test),'MAE':np.mean(abs(test-yhat)),'RMSE':np.sqrt(np.mean((test-yhat)**2)),'Training ADF p':adfuller(train)[1],'Tra

/tmp/ipykernel_835/984586289.py:23: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  results.append({'Company':a,'Method':label,'Order':str((p,d,q)),'Test N':len(test),'MAE':np.mean(abs(test-yhat)),'RMSE':np.sqrt(np.mean((test-yhat)**2)),'Training ADF p':adfuller(train)[1],'Training KPSS p':kpss(train,nlags='auto')[1],'Residual Ljung Box p':acorr_ljungbox(best.resid,lags=[10],model_df=p+q,return_df=True).lb_pvalue.iloc[0],'Residual ARCH LM p':het_arch(best.resid,nlags=10)[1]})
/tmp/ipykernel_835/984586289.py:23: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  results.append({'Company':a,'Method':label,'Order':str((p,d,q)),'Test N':len(test),'MAE':np.mean(abs(test-yhat)),'RMSE':np.sqrt(np.mean((test-yhat)**2)),'Training ADF p':adfuller(train)[1],'Tra

/tmp/ipykernel_835/984586289.py:23: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  results.append({'Company':a,'Method':label,'Order':str((p,d,q)),'Test N':len(test),'MAE':np.mean(abs(test-yhat)),'RMSE':np.sqrt(np.mean((test-yhat)**2)),'Training ADF p':adfuller(train)[1],'Training KPSS p':kpss(train,nlags='auto')[1],'Residual Ljung Box p':acorr_ljungbox(best.resid,lags=[10],model_df=p+q,return_df=True).lb_pvalue.iloc[0],'Residual ARCH LM p':het_arch(best.resid,nlags=10)[1]})
/tmp/ipykernel_835/984586289.py:23: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  results.append({'Company':a,'Method':label,'Order':str((p,d,q)),'Test N':len(test),'MAE':np.mean(abs(test-yhat)),'RMSE':np.sqrt(np.mean((test-yhat)**2)),'Training ADF p':adfuller(train)[1],'Tra

,Company,p,d,q,AIC,BIC,Converged
0,JKH,0,0,0,8951.752571,8963.298919,True
1,JKH,1,0,0,8925.703989,8943.023510,True
2,JKH,0,0,1,8927.775685,8945.095206,True
3,JKH,1,0,1,8924.160768,8947.253462,True
4,JKH,2,0,0,8924.067828,8947.160523,True
5,JKH,0,0,2,8925.108205,8948.200900,True
6,HAYL,0,0,0,10461.811624,10473.184866,True
7,HAYL,1,0,0,10462.444289,10479.504153,True
8,HAYL,0,0,1,10462.415347,10479.475211,True
9,HAYL,1,0,1,10462.323413,10485.069898,True


,Company,Method,Order,Test N,MAE,RMSE,Training ADF p,Training KPSS p,Residual Ljung Box p,Residual ARCH LM p
0,JKH,ARIMA,"(2, 0, 0)",594,0.963738,1.371754,0.000000e+00,0.1,0.000003,1.596315e-72
1,JKH,Zero return,"(2, 0, 0)",594,0.953861,1.374560,0.000000e+00,0.1,0.000003,1.596315e-72
2,HAYL,ARIMA,"(0, 0, 0)",545,1.106979,1.649826,5.777156e-17,0.1,0.000001,5.830678e-39
3,HAYL,Zero return,"(0, 0, 0)",545,1.104662,1.653813,5.777156e-17,0.1,0.000001,5.830678e-39
4,VONE,ARIMA,"(2, 0, 0)",582,1.245804,1.754271,8.593245e-18,0.1,0.000002,1.343991e-52
5,VONE,Zero return,"(2, 0, 0)",582,1.216461,1.734790,8.593245e-18,0.1,0.000002,1.343991e-52


## Interpretation
Lower test RMSE/MAE indicates better performance on the same company and holdout. A significant residual ARCH test motivates conditional variance modelling rather than assuming constant variance. A return forecast is not a volatility forecast. Order selection is restricted to the stated candidate set. Corporate action verification remains outstanding.